In [35]:
from pathlib import Path
import sys
import copy
import pandas as pd
import yaml

from pycap.analysis_project import Project

# from repo scripts
scripts_dir = Path.cwd().parent / "scripts"
print("scripts_dir:", scripts_dir)
sys.path.insert(0, str(scripts_dir))

if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from pycap_for_PESTPP_MOU import postprocess_MOU


scripts_dir: /workspaces/LPR_redux/LPR_pycap_opt/scripts


In [36]:
dirs_to_check = [
    Path("/workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/depletion_q_baseline_0.0_1.0_0.2_template"),
    Path("/workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/run_depletion_q_baseline_0.0_1.0_0.2"),
    Path("/workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/fish_dollars_baseline_0.0_1.0_0.2_template"),
    Path("/workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/run_fish_dollars_baseline_0.0_1.0_0.2"),
]

for d in dirs_to_check:
    print("\n" + "="*90)
    print(f"Directory: {d}")
    print(f"Exists: {d.exists()}")
    
    if d.exists():
        items = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        print(f"Number of items: {len(items)}\n")
        
        for item in items:
            kind = "[DIR]" if item.is_dir() else "[FILE]"
            print(f"{kind:6} {item.name}")


Directory: /workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/depletion_q_baseline_0.0_1.0_0.2_template
Exists: True
Number of items: 15

[FILE] allobs.out
[FILE] allobs.out.ins
[FILE] basedeplobs.dat
[FILE] depletion_q_baseline_0.0_1.0_0.2.insfile_data.csv
[FILE] depletion_q_baseline_0.0_1.0_0.2.obs_data.csv
[FILE] depletion_q_baseline_0.0_1.0_0.2.par_data.csv
[FILE] depletion_q_baseline_0.0_1.0_0.2.pargp_data.csv
[FILE] depletion_q_baseline_0.0_1.0_0.2.pi_data.csv
[FILE] depletion_q_baseline_0.0_1.0_0.2.pst
[FILE] depletion_q_baseline_0.0_1.0_0.2.tplfile_data.csv
[FILE] initial_dvpop.csv
[FILE] LPR_Redux.yml
[FILE] LPR_Redux.yml.tpl
[FILE] pestpp-mou
[FILE] run_pycap_standalone_opt_mou.py

Directory: /workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/run_depletion_q_baseline_0.0_1.0_0.2
Exists: True
Number of items: 333

[FILE] allobs.out.ins
[FILE] basedeplobs.dat
[FILE] depletion_q_baseline_0.0_1.0_0.2.0.archive.dv_pop.csv.zip
[FILE] depletion_q_baseline_0.0_1.0_0.2.

In [37]:
template_dir = Path("/workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/depletion_q_baseline_0.0_1.0_0.2_template")
run_dir = Path("/workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/run_depletion_q_baseline_0.0_1.0_0.2")

template_names = {p.name for p in template_dir.iterdir()}
run_names = {p.name for p in run_dir.iterdir()}

only_in_template = sorted(template_names - run_names)
only_in_run = sorted(run_names - template_names)
in_both = sorted(template_names & run_names)

print("="*90)
print("Files only in template directory:")
for name in only_in_template:
    print(name)

print("\n" + "="*90)
print("Files only in run directory:")
for name in only_in_run:
    print(name)

print("\n" + "="*90)
print("Files in both directories:")
for name in in_both:
    print(name)

Files only in template directory:
allobs.out

Files only in run directory:
depletion_q_baseline_0.0_1.0_0.2.0.archive.dv_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.0.archive.obs_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.0.archive.pi_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.0.dv_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.0.obs_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.1.archive.dv_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.1.archive.obs_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.1.archive.pi_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.1.dv_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.1.obs_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.1.pi_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.10.archive.dv_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.10.archive.obs_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.10.archive.pi_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.10.dv_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.10.obs_pop.csv.zip
depletion_q_baseline_0.0_1.0_0.2.10.pi_pop

In [38]:
run_name = "depletion_q_baseline_0.0_1.0_0.2"   # or your actual baseline scenario name
run_path = Path(f"../pycap_runs/pycap_pest/run_{run_name}")

pareto_df = postprocess_MOU(run_name, run_path)
root_dir = Path.cwd().parent

# baseline depletion-vs-pumping run
run_name = "depletion_q_baseline_0.0_1.0_0.2"
run_path = root_dir / "pycap_runs" / "pycap_pest" / f"run_{run_name}"

# where to save the new outputs
output_dir = Path.cwd() / "pareto_member_T_runs"
output_dir.mkdir(exist_ok=True)

# transmissivity scenarios
t_scenarios = {
    "Tplus10per": 1.10,
    "Tmin10per": 0.90,
}

print("run_path:", run_path)
print("exists:", run_path.exists())
print("output_dir:", output_dir)
# usually the final generation front
final_generation = pareto_df["generation"].max()
pareto_df_final = pareto_df.loc[pareto_df["generation"] == final_generation].copy()

pareto_members = pareto_df_final["member"].astype(str).tolist()
pareto_df_final[["member"]].head()

run_path: /workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/run_depletion_q_baseline_0.0_1.0_0.2
exists: True
output_dir: /workspaces/LPR_redux/LPR_pycap_opt/notebooks/pareto_member_T_runs


,member
5452,10
5453,gen=49_member=1825_pso
5454,12
5455,36
5456,gen=43_member=1604_pso


In [39]:
root_dir = Path.cwd().parent

# baseline depletion-vs-pumping run
run_name = "depletion_q_baseline_0.0_1.0_0.2"
run_path = root_dir / "pycap_runs" / "pycap_pest" / f"run_{run_name}"

# where to save the new outputs
output_dir = Path.cwd() / "pareto_member_T_runs"
output_dir.mkdir(exist_ok=True)

# transmissivity scenarios
t_scenarios = {
    "Tplus10per": 1.10,
    "Tmin10per": 0.90,
}

print("run_path:", run_path)
print("exists:", run_path.exists())
print("output_dir:", output_dir)

run_path: /workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/run_depletion_q_baseline_0.0_1.0_0.2
exists: True
output_dir: /workspaces/LPR_redux/LPR_pycap_opt/notebooks/pareto_member_T_runs


In [40]:
def load_pareto_archive_summary(run_name, run_path):
    candidate_files = [
        run_path / f"{run_name}.pareto.archive.summary.csv.zip",
        run_path / f"{run_name}.pareto.archive.summary.csv",
    ]

    for f in candidate_files:
        if f.exists():
            return pd.read_csv(f), f

    raise FileNotFoundError(
        "Could not find either of these files:\n"
        + "\n".join(str(f) for f in candidate_files)
    )


def load_final_feasible_pareto_front(run_name, run_path):
    pareto_df, pareto_file = load_pareto_archive_summary(run_name, run_path)

    # keep only feasible, non-dominated solutions
    pareto_df = pareto_df.loc[
        (pareto_df["nsga2_front"] == 1) &
        (pareto_df["is_feasible"] == 1)
    ].copy()

    pareto_df["member"] = pareto_df["member"].astype(str)

    final_generation = pareto_df["generation"].max()
    pareto_df_final = pareto_df.loc[
        pareto_df["generation"] == final_generation
    ].copy()

    return pareto_df_final, pareto_file, final_generation

In [41]:
pareto_df_final, pareto_file, final_generation = load_final_feasible_pareto_front(
    run_name,
    run_path
)

pareto_members = pareto_df_final["member"].tolist()

print("Loaded pareto summary from:", pareto_file.name)
print("Final generation:", final_generation)
print("Number of final Pareto members:", len(pareto_members))

pareto_df_final[["member", "generation"]].head()

Loaded pareto summary from: depletion_q_baseline_0.0_1.0_0.2.pareto.archive.summary.csv.zip
Final generation: 50
Number of final Pareto members: 179


,member,generation
5452,10,50
5453,gen=49_member=1825_pso,50
5454,12,50
5455,36,50
5456,gen=43_member=1604_pso,50


In [42]:
def load_decision_variable_population(run_path):
    dv_files = sorted(run_path.glob("*dv_pop.csv")) + sorted(run_path.glob("*dv_pop.csv.zip"))

    if not dv_files:
        raise FileNotFoundError(f"No dv_pop files found in {run_path}")

    frames = [pd.read_csv(f, index_col=0) for f in dv_files]

    init_file = run_path / "initial_dvpop.csv"
    if init_file.exists():
        frames.append(pd.read_csv(init_file, index_col=0))

    dv_df = pd.concat(frames, axis=0)
    dv_df.index = dv_df.index.astype(str)
    dv_df = dv_df[~dv_df.index.duplicated(keep="first")]

    return dv_df


dv_df = load_decision_variable_population(run_path)

missing_members = [m for m in pareto_members if m not in dv_df.index]
if missing_members:
    raise KeyError(
        f"{len(missing_members)} Pareto members were not found in dv_pop files. "
        f"First few missing: {missing_members[:5]}"
    )

member_q_df = dv_df.loc[pareto_members].copy()

print("member_q_df shape:", member_q_df.shape)
member_q_df.head()

member_q_df shape: (179, 327)


,well_1013__q,well_1302__q,well_1323__q,well_1486__q,well_1584__q,well_1589__q,well_1643__q,well_1683__q,well_1860__q,well_23610__q,...,well_93143__q,well_93349__q,well_93422__q,well_93423__q,well_93424__q,well_93469__q,well_93832__q,well_94302__q,well_94988__q,well_95068__q
10,125.1,88.2,76.2,222.8,150.5,264.1,114.5,260.6,195.7,201.7,...,0.6,27.2,0.1,0.1,0.2,158.2,14.6,148.9,11.1,40.3
gen=49_member=1825_pso,125.1,88.2,76.2,222.8,150.5,264.1,114.5,260.6,195.7,201.7,...,0.6,27.2,0.1,0.1,0.2,158.2,14.6,148.9,11.1,40.3
12,125.1,88.2,76.2,222.8,150.5,264.1,114.5,260.6,195.7,201.7,...,0.6,27.2,0.1,0.1,0.2,158.2,14.6,148.9,11.1,40.3
36,125.1,88.2,76.2,222.8,150.5,264.1,114.5,260.6,195.7,201.7,...,0.6,27.2,0.1,0.1,0.2,158.2,14.6,148.9,11.1,40.3
gen=43_member=1604_pso,125.1,88.2,76.2,222.8,150.5,264.1,114.5,260.6,195.7,201.7,...,0.6,27.2,0.1,0.1,0.2,158.2,14.6,148.9,11.1,40.3


In [43]:
with open(run_path / "LPR_Redux.yml", "r") as f:
    base_dict = yaml.safe_load(f)

print("Base T:", base_dict["project_properties"]["T"])


def get_results_with_T(pars, initial_dict, bdplobs=None, T_multiplier=1.0, write_csv=False):
    """
    Re-evaluate one pumping realization with a modified transmissivity.
    pars should be a Series whose index looks like well_###__q
    """
    pars = pars.copy()
    pars.index = [i.lower() for i in pars.index]

    qpars = pars.loc[pars.index.str.contains("_q")]

    # deep copy so nested dicts do not get overwritten between members
    upd_dict = copy.deepcopy(initial_dict)

    # update transmissivity
    upd_dict["project_properties"]["T"] = initial_dict["project_properties"]["T"] * T_multiplier

    # optional: keep bounds consistent in the copied dict
    if "Max_T" in upd_dict["project_properties"]:
        upd_dict["project_properties"]["Max_T"] = initial_dict["project_properties"]["Max_T"] * T_multiplier
    if "Min_T" in upd_dict["project_properties"]:
        upd_dict["project_properties"]["Min_T"] = initial_dict["project_properties"]["Min_T"] * T_multiplier

    # update pumping for this member
    for idx, val in qpars.items():
        well_key = idx.split("__")[0]
        upd_dict[well_key]["Q"] = val

    ap = Project(None, write_csv, upd_dict)
    ap.aggregate_results()

    bdf = ap.agg_base_stream_df.copy()
    bdf.index = [f"lpr:{i}:bdpl" for i in bdf.index]
    bdf["variable"] = bdf.index
    bdf.rename(columns={"LPR": "value"}, inplace=True)

    if bdplobs is not None:
        bdf = bdf.loc[bdplobs]

    return bdf["value"]

Base T: 1700.0


In [44]:
scenario_results = {}

for scenario_name, T_multiplier in t_scenarios.items():
    print(f"Running {scenario_name}  (T x {T_multiplier})")

    scenario_member_results = {}

    for member, row in member_q_df.iterrows():
        scenario_member_results[member] = get_results_with_T(
            row,
            base_dict,
            T_multiplier=T_multiplier,
            write_csv=False
        )

    scenario_df = pd.DataFrame(scenario_member_results).T
    scenario_df.index.name = "member"
    scenario_df["scenario"] = scenario_name
    scenario_df["T_multiplier"] = T_multiplier

    scenario_results[scenario_name] = scenario_df

results_plus10_df = scenario_results["Tplus10per"].copy()
results_minus10_df = scenario_results["Tmin10per"].copy()

print("results_plus10_df shape:", results_plus10_df.shape)
print("results_minus10_df shape:", results_minus10_df.shape)

Running Tplus10per  (T x 1.1)


AttributeError: 'Project' object has no attribute 'agg_base_stream_df'

In [ ]:
pareto_meta = pareto_df_final.copy()
pareto_meta["member"] = pareto_meta["member"].astype(str)
pareto_meta = pareto_meta.set_index("member")

results_plus10_full_df = pareto_meta.join(results_plus10_df, how="left")
results_minus10_full_df = pareto_meta.join(results_minus10_df, how="left")

combined_results_df = pd.concat(
    [results_plus10_full_df, results_minus10_full_df],
    axis=0
).reset_index()

print("results_plus10_full_df shape:", results_plus10_full_df.shape)
print("results_minus10_full_df shape:", results_minus10_full_df.shape)
print("combined_results_df shape:", combined_results_df.shape)

results_plus10_full_df.head()

In [ ]:
member_q_df.to_csv(output_dir / f"{run_name}_pareto_members_q.csv")

results_plus10_full_df.reset_index().to_csv(
    output_dir / f"{run_name}_Tplus10per_reevaluated.csv",
    index=False
)

results_minus10_full_df.reset_index().to_csv(
    output_dir / f"{run_name}_Tmin10per_reevaluated.csv",
    index=False
)

combined_results_df.to_csv(
    output_dir / f"{run_name}_Tplus10per_Tmin10per_reevaluated_combined.csv",
    index=False
)

print("Saved files:")
for f in sorted(output_dir.glob(f"{run_name}*.csv")):
    print(f.name)

In [ ]:
print("Plus 10% T:")
print(results_plus10_full_df.iloc[:5, :10])

print("\nMinus 10% T:")
print(results_minus10_full_df.iloc[:5, :10])